# DS-GA 1019

# Lab 6 

## Part 1

## Feb. 26, 2026

## Cuda

A general purpose parallel computing platform and programming model

<img  src="https://i.ibb.co/JQGz39k/cuda1.png" alt="Drawing" style="width: 400px;"/>


- Grid: Group of blocks executing a kernel (One grid per CUDA kernel)

- Block: Group of threads that can be scheduled independently (Max threads in a block: 1024)
- Thread: A single context of execution
- Kernel: Function that will execute in parallel on multiple threads


<img src="https://i.ibb.co/jkyFsvc/cuda2.png" alt="Drawing" style="width: 700px;"/>

<img  src="https://i.ibb.co/hFp1yTc/cuda3.png" alt="Drawing" style="width: 400px;"/>

CUDA Memory Hierarchy:
- Per thread local memory: Available only to the single thread

- Block shared memory: Shared by all the threads in a block (Faster than global memory)
- Global memory: Shared by all blocks and all grids


## Global memory access

<img src="https://i.ibb.co/rQ4sLvg/matmul1.png" alt="Drawing" style="width: 800px;"/>

In [1]:
%env NUMBA_ENABLE_CUDASIM=1

env: NUMBA_ENABLE_CUDASIM=1


In [2]:
from numba import cuda
print(cuda.gpus)

<Managed Device 0>


## A more complex example: matrix multiplication

Naive matrix multiplication using a global memory is defined in `spring2026_lab06_part01.ipynb`'s `matmul(...)` function. Refer to it for details.

We previously discussed using global memory is ineifficient.

### Shared memory and thread synchronization
A limited amount of shared memory can be allocated on the device to speed up access to data, when necessary. That memory will be shared (i.e. both readable and writable) amongst all threads belonging to a given block and has faster access times than regular device memory. It also allows threads to cooperate on a given solution. You can think of it as a manually-managed data cache.

#### Creating shared memory array
- The function to create a shared memory array is: `shared_array = cuda.shared.array(shape,type)`
  - **shape** is either an integer or a tuple of integers representing the array’s dimensions.

  - **type** is a Numba type of the elements needing to be stored in the array. 

- The memory is allocated once for the **duration of the kernel**, unlike traditional dynamic memory management.

- Because the shared memory is a limited resource, it is often necessary to preload a small block at a time from the input arrays. All the threads then need to wait until everyone has finished preloading before doing the computation on the shared memory.

#### Synchronization

- Synchronization is then required again after the computation to ensure all threads have finished with the data in shared memory before overwriting it in the next loop iteration.

- The function to synchronized threads is: `cuda.syncthreads()`
  - It will synchronize all threads in the same thread block. This function implements the same pattern as barriers in traditional multi-threaded programming and the ```MPI.Barrier()``` function. The program will wait until all threads in the block call the function, at which point it returns control to all its callers.

## Improved matrix multiplication (Using Shared Memory)
The following example shows how shared memory can be used when performing matrix multiplication.

In this example, each thread block is responsible for computing a square sub-matrix of `C` and each thread for computing an element of the sub-matrix. The sub-matrix is equal to the product of a square sub-matrix of `A` (`sA`) and a square sub-matrix of `B` (`sB`). In order to fit into the device resources, the two input matrices are divided into as many square sub-matrices of dimension `TPB` as necessary, and the result computed as the sum of the products of these square sub-matrices.

Each product is performed by first loading sA and sB from global memory to shared memory, with one thread loading each element of each sub-matrix. Once sA and sB have been loaded, each thread accumulates the result into a register (`tmp`). Once all the products have been calculated, the results are written to the matrix `C` in global memory.

By blocking the computation this way, we can reduce the number of global memory accesses since `A` is now only read `B.shape[1] / TPB` times and `B` is read `A.shape[0] / TPB` times.

<img align="left"  src="https://i.ibb.co/5xZ5cys/improve-matmul.png" alt="Drawing" style="width: 400px;"/>

In [5]:
from numba import cuda, float32
import numpy as np
import math

# Controls threads per block and shared memory usage.
# The computation will be done on blocks of TPBxTPB elements.
TPB = 16

@cuda.jit
def fast_matmul(A, B, C):
    """
    Perform matrix multiplication of C = A * B
    Each thread computes one element of the result matrix C
    """

    # Define an array in the shared memory
    # The size and type of the arrays must be known at compile time
    sA = cuda.shared.array(shape=(TPB, TPB), dtype=float32)
    sB = cuda.shared.array(shape=(TPB, TPB), dtype=float32)

    x, y = cuda.grid(2)
    # print(x, y)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    if x >= C.shape[0] and y >= C.shape[1]:
        # Quit if (x, y) is outside of valid C boundary
        return

    # Each thread computes one element in the result matrix.
    # The dot product is chunked into dot products of TPB-long vectors.
    tmp = 0.
    for i in range(int(A.shape[1] / TPB)):
        # Preload data into shared memory
        sA[tx, ty] = A[x, ty + i * TPB]
        sB[tx, ty] = B[tx + i * TPB, y]

        # Wait until all threads finish preloading
        cuda.syncthreads()

        # Computes partial product on the shared memory
        for j in range(TPB):
            tmp += sA[tx, j] * sB[j, ty]

        # Wait until all threads finish computing
        cuda.syncthreads()

    C[x, y] = tmp

# The data array
A = np.full((TPB*2, TPB*3), 3, np.float64) # [32 x 48] matrix containing all 3's
B = np.full((TPB*3, TPB*1), 4, np.float64) # [48 x 16] matrix containing all 4's

A_global_mem = cuda.to_device(A)
B_global_mem = cuda.to_device(B)
C_global_mem = cuda.device_array((TPB*2, TPB*1)) # [32 x 16] matrix result

# Configure the blocks
threadsperblock = (TPB, TPB)
blockspergrid_x = int(math.ceil(A.shape[0] / threadsperblock[1]))
blockspergrid_y = int(math.ceil(B.shape[1] / threadsperblock[0]))
blockspergrid = (blockspergrid_x, blockspergrid_y)

# Start the kernel
fast_matmul[blockspergrid, threadsperblock](A_global_mem, B_global_mem, C_global_mem)
res = C_global_mem.copy_to_host()

print(res)

[[576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576.
  576. 576.]
 [576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576. 576